# The Local Smart Coding Tutor

This project combines a cloud LLM (via OpenRouter) with three local Hugging Face pipelines, all served through a Gradio UI:
- **Whisper** (`openai/whisper-small`) — transcribes microphone input locally.
- **Zero-Shot Classification** (`facebook/bart-large-mnli`) — classifies user intent locally to shape the system prompt *before* the LLM API call.
- **SpeechT5** (`microsoft/speecht5_tts`) — synthesizes the LLM's response into speech locally.

It's a slight improvement to the project I submitted in week2

In [ ]:
!pip install -q --upgrade transformers==4.57.6 datasets==3.6.0 soundfile

In [ ]:
import json
import os

import gradio as gr
import numpy as np
import requests
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from transformers import pipeline

load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key
)
AGENT_MODEL = "openai/gpt-4o-mini"

## 1. Pre-loading the Hugging Face Local Pipelines
On Apple Silicon Macs, models use the `mps` backend; on NVIDIA GPUs they use `cuda`; otherwise they fall back to `cpu`.

In [ ]:
# Ensure you have huggingface token set in .env or Google Colab secrets.
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps" # support for Apple Silicon Macs
else:
    device = "cpu"
print(f"Using device: {device}")

print("Loading Whisper...")
asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=device)

print("Loading Zero-Shot Classifier...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

print("Loading SpeechT5 (Text-to-Speech)...")
tts_pipe = pipeline("text-to-speech", "microsoft/speecht5_tts", device=device)
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

print("All local models loaded successfully!")

## 2. Defining The Tools (From Week 2)

In [ ]:
def fetch_proverb_chapter(chapter_number):
    response = requests.get(f"https://bible-api.com/proverbs+{chapter_number}")
    if response.status_code == 200:
        return f"Text from Proverbs Chapter {chapter_number}:\n" + response.json().get('text', '')
    return f"Failed to fetch chapter {chapter_number}."

def extract_football_analogy(pattern_name, team_or_player):
    return f"""
    You are an expert software engineer and a huge football fan.
    Explain '{pattern_name}' using a tactical analogy based on '{team_or_player}'.
    """

def execute_tool(name, args):
    if name == "fetch_proverb_chapter":
        return fetch_proverb_chapter(args.get("chapter_number", 1))
    elif name == "extract_football_analogy":
        return extract_football_analogy(args.get("pattern_name"), args.get("team_or_player"))
    return "Unknown tool."

tools = [
    {"type": "function", "function": {
        "name": "fetch_proverb_chapter",
        "description": "Fetch raw text from a Proverbs chapter.",
        "parameters": {"type": "object", "properties": {"chapter_number": {"type": "integer"}}, "required": ["chapter_number"]}
    }},
    {"type": "function", "function": {
        "name": "extract_football_analogy",
        "description": "Instruct LLM to explain a coding pattern using football tactics.",
        "parameters": {"type": "object", "properties": {"pattern_name": {"type": "string"}, "team_or_player": {"type": "string"}}, "required": ["pattern_name", "team_or_player"]}
    }}
]

## 3. The Smart Zero-Shot Context Router

In [ ]:
def smart_route_intent(user_text):
    """
    Classifies the user text locally before sending anything to the LLM.
    If user is asking about coding, we prepend a strict Persona instruction.
    If it's bible reading, we relax the persona.
    """
    labels = ["software engineering and coding", "bible verses and religion", "casual conversation"]
    result = classifier(user_text, candidate_labels=labels)
    top_intent = result['labels'][0]
    
    print(f"Zero-Shot Router classified input as: {top_intent} ({result['scores'][0]*100:.1f}% confidence)")
    
    system_instruction = "You are Toluwalemi's Universal Assistant. Use your tools if requested.\n"
    if top_intent == "software engineering and coding":
        system_instruction += "[ROUTER DETECTED CODING]: Be highly technical, concise, use markdown code blocks, and lean heavily into tactical football metaphors if applicable.\n"
    elif top_intent == "bible verses and religion":
        system_instruction += "[ROUTER DETECTED THEOLOGY]: Be poetic, respectful, and philosophical. Extract the direct lesson.\n"
    
    return system_instruction

## 4. The Agent Streaming Loop & UI Logic

In [ ]:
def parse_audio_input(audio_path):
    """Converts microphone audio into text using local Whisper pipeline"""
    if audio_path is None:
        return ""
    print("Transcribing Audio locally...")
    result = asr_pipe(audio_path)
    return result["text"]

def start_chat(text_input, audio_input, history):
    # If audio is provided, extract it to text. Audio overrides typing.
    true_input = text_input
    if audio_input:
        true_input = parse_audio_input(audio_input)
    
    if not true_input.strip():
        return "", None, history

    return "", None, history + [{"role": "user", "content": true_input}]

def generate_agent_response(history):
    if not history or history[-1]["role"] != "user":
        yield history, None
        return
        
    user_input = history[-1]["content"]

    # SMART ROUTING
    system_message = smart_route_intent(user_input)

    # OPENROUTER LOOP
    backend_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + backend_history
    history.append({"role": "assistant", "content": ""})
    
    try:
        while True:
            stream = openrouter.chat.completions.create(
                model=AGENT_MODEL, messages=messages, tools=tools, stream=True
            )
            reply_text = ""
            tool_calls_by_index = {}
            finish_reason = None

            for chunk in stream:
                if not chunk.choices:
                    continue
                choice = chunk.choices[0]
                finish_reason = choice.finish_reason or finish_reason
                delta = choice.delta
                if delta and delta.content:
                    reply_text += delta.content
                    history[-1]["content"] = reply_text
                    yield history, None

                if delta and delta.tool_calls:
                    for tc_delta in delta.tool_calls:
                        idx = tc_delta.index
                        if idx not in tool_calls_by_index:
                            tool_calls_by_index[idx] = {"id": "", "name": "", "arguments": ""}
                        if tc_delta.id:
                            tool_calls_by_index[idx]["id"] = tc_delta.id
                        if tc_delta.function and tc_delta.function.name:
                            tool_calls_by_index[idx]["name"] = tc_delta.function.name
                        if tc_delta.function and tc_delta.function.arguments:
                            tool_calls_by_index[idx]["arguments"] += tc_delta.function.arguments

            if finish_reason != "tool_calls" or not tool_calls_by_index:
                break
                
            assistant_msg = {
                "role": "assistant",
                "tool_calls": [
                    {"id": data["id"], "type": "function", "function": {"name": data["name"], "arguments": data["arguments"]}}
                    for data in tool_calls_by_index.values()
                ]
            }
            messages.append(assistant_msg)

            for data in tool_calls_by_index.values():
                result = execute_tool(data["name"], json.loads(data["arguments"]))
                messages.append({"role": "tool", "content": result, "tool_call_id": data["id"]})
    except Exception as e:
        print(f"LLM API call failed: {e}")
        history[-1]["content"] = f"Sorry, the LLM call failed: {e}"
        yield history, None
        return

    # Local SpeechT5 TTS
    print("Generating local audio from text...")
    try:
        tts_text = reply_text[:500] if len(reply_text) > 500 else reply_text
        speech = tts_pipe(tts_text, forward_params={"speaker_embeddings": speaker_embedding})
        audio_output = (speech["sampling_rate"], np.array(speech["audio"]))
    except Exception as e:
        print(f"Audio failed: {e}")
        audio_output = None

    yield history, audio_output

with gr.Blocks(theme=gr.themes.Soft()) as ui:
    gr.Markdown("# Local 'Smart' Tutor Assistant")
    gr.Markdown("*Powered by OpenRouter LLM, local Whisper Speech, local Zero-Shot routing, and local SpeechT5 TTS.*")
    
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=500, type="messages")
            message_input = gr.Textbox(label="Type here...", placeholder="e.g. Can you explain 2-pointers using an Arsenal analogy?")
            audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Or Speak your prompt!")
            submit_btn = gr.Button("Send")
        with gr.Column(scale=1):
            audio_output = gr.Audio(label="Voice Output", autoplay=True)

    for trigger in [message_input.submit, submit_btn.click]:
        trigger(
            start_chat, 
            inputs=[message_input, audio_input, chatbot], 
            outputs=[message_input, audio_input, chatbot]
        ).then(
            generate_agent_response, 
            inputs=chatbot, 
            outputs=[chatbot, audio_output]
        )

if __name__ == "__main__":
    ui.launch(inbrowser=True)